<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week4/week4_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Week 4 Header & Library Imports
# Eye of Ra 👁️ — Week 4: Streamlit Dashboard & Technical Abstract

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries loaded — Week 4 begins")
print("👁️ Eye of Ra — Dashboard & Submission")

# Git: Cell 1 — Week 4 header and imports

✅ Libraries loaded — Week 4 begins
👁️ Eye of Ra — Dashboard & Submission


In [3]:
# Cell 2 — Load dataset and scored contracts
df = pd.read_csv('/content/eye_of_ra_master_dataset_v4.csv')
scored = pd.read_csv('/content/eye_of_ra_scored_contracts_v2.csv')
model = joblib.load('/content/eye_of_ra_model_v1.pkl')
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')

print(f"✅ Master dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Scored contracts: {scored.shape[0]} rows x {scored.shape[1]} columns")
print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Features loaded: {len(feature_cols)} features")
print(f"\nTier breakdown:")
print(scored['tier'].value_counts())

# Git: Cell 2 — load all files for Week 4

✅ Master dataset: 262 rows x 21 columns
✅ Scored contracts: 262 rows x 14 columns
✅ Model loaded: XGBClassifier
✅ Features loaded: 12 features

Tier breakdown:
tier
🟢 MONITOR     241
🔴 ESCALATE     18
🟡 REVIEW        3
Name: count, dtype: int64


In [ ]:
# Cell 3 — Install Streamlit and ngrok
!pip install streamlit pyngrok -q

from pyngrok import ngrok

print("✅ Streamlit installed")
print("✅ pyngrok installed")

# Git: Cell 3 — install Streamlit and ngrok

✅ Streamlit installed
✅ pyngrok installed


In [ ]:
# Cell 4 — Write the Streamlit dashboard file

dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Page config
st.set_page_config(
    page_title="Eye of Ra 👁️",
    page_icon="👁️",
    layout="wide"
)

# Load data
@st.cache_data
def load_data():
    scored = pd.read_csv("/content/eye_of_ra_scored_contracts_v1.csv")
    return scored

scored = load_data()

# Clean up flags column for display
scored["flags"] = scored["flags"].fillna("[]")

# ── SIDEBAR ──────────────────────────────────────────────
st.sidebar.image("https://flagcdn.com/w80/gh.png", width=60)
st.sidebar.title("👁️ Eye of Ra")
st.sidebar.markdown("AI Procurement Fraud Detection for Ghana")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])
st.sidebar.markdown("---")
st.sidebar.markdown("Built on Ghana PPA & Auditor-General data")
st.sidebar.markdown("Grounded in Public Procurement Act 663")

# ── PAGE 1: OVERVIEW ─────────────────────────────────────
if page == "📊 Overview":
    st.title("👁️ Eye of Ra — Procurement Fraud Detection")
    st.markdown("### Ghana AI Summit 2026 | Real Data. Real Flags. Real Accountability.")
    st.markdown("---")

    # Top metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col3.metric("🔴 Escalate", str(len(scored[scored["tier"] == "🔴 ESCALATE"])))
    col4.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("---")

    col1, col2 = st.columns(2)

    # Tier distribution pie
    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#d32f2f",
                "🟡 REVIEW": "#f9a825",
                "🟢 MONITOR": "#388e3c"
            }
        )
        st.plotly_chart(fig, use_container_width=True)

    # Fraud detection bar
    with col2:
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\\n(17 caught)", "🟡 REVIEW\\n(3 caught)", "🟢 MONITOR\\n(3 missed)"],
            y=[17, 3, 3],
            marker_color=["#d32f2f", "#f9a825", "#388e3c"],
            text=[17, 3, 3],
            textposition="outside"
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            yaxis_title="Number of Contracts",
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    # Score distribution
    st.markdown("### Score Distribution — Fraud vs Clean")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#388e3c", 1: "#d32f2f"},
        labels={"fraud_label": "Fraud", "composite_score": "Composite Risk Score"},
        title="Risk Score Distribution"
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#d32f2f", annotation_text="ESCALATE threshold")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f9a825", annotation_text="REVIEW threshold")
    st.plotly_chart(fig3, use_container_width=True)

# ── PAGE 2: FLAGGED CONTRACTS ─────────────────────────────
elif page == "🔴 Flagged Contracts":
    st.title("🔴 Flagged Contracts")
    st.markdown("Contracts in ESCALATE and REVIEW tiers — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False)

    st.markdown(f"**{len(filtered)} contracts shown**")
    st.dataframe(
        filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 3: ENTITY SCORECARDS ─────────────────────────────
elif page == "🏛️ Entity Scorecards":
    st.title("🏛️ Entity Integrity Scorecards")
    st.markdown("Search any procurement entity and see their full risk profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Select an entity:", entity_list)

    entity_contracts = scored[scored["entity"] == selected_entity]
    max_score = entity_contracts["composite_score"].max()
    fraud_count = entity_contracts["fraud_label"].sum()
    total = len(entity_contracts)
    escalate_count = len(entity_contracts[entity_contracts["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Contracts", total)
    col2.metric("Max Risk Score", f"{max_score:.1f}")
    col3.metric("Confirmed Fraud", int(fraud_count))
    col4.metric("Escalated", escalate_count)

    st.markdown("---")
    st.markdown("### All contracts for this entity:")
    st.dataframe(
        entity_contracts[["supplier", "composite_score", "tier", "fraud_label"]].sort_values("composite_score", ascending=False).reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 4: CONTRACT DEEP DIVE ────────────────────────────
elif page == "🔍 Contract Deep Dive":
    st.title("🔍 Contract Deep Dive")
    st.markdown("Select any contract for a full risk breakdown.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract index:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:50]}"
    )

    row = scored.loc[contract_idx]

    col1, col2 = st.columns(2)
    with col1:
        st.markdown(f"**Entity:** {row['entity']}")
        st.markdown(f"**Supplier:** {row['supplier']}")
        st.markdown(f"**Composite Score:** {row['composite_score']}")
        st.markdown(f"**Tier:** {row['tier']}")
        st.markdown(f"**Fraud Confirmed:** {'✅ Yes' if row['fraud_label'] == 1 else '⬜ No'}")

    with col2:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=row["composite_score"],
            title={"text": "Risk Score"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#d32f2f" if row["composite_score"] >= 50 else "#f9a825" if row["composite_score"] >= 35 else "#388e3c"},
                "steps": [
                    {"range": [0, 35], "color": "#e8f5e9"},
                    {"range": [35, 50], "color": "#fff9c4"},
                    {"range": [50, 100], "color": "#ffebee"}
                ],
                "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.75, "value": row["composite_score"]}
            }
        ))
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("### Flags raised:")
    flags = eval(row["flags"]) if isinstance(row["flags"], str) else row["flags"]
    if flags:
        for f in flags:
            st.markdown(f"• {f}")
    else:
        st.markdown("No flags raised.")

    st.markdown("### Legal citations:")
    citations = eval(row["legal_citations"]) if isinstance(row["legal_citations"], str) else row["legal_citations"]
    if citations:
        for c in citations:
            st.markdown(f"• {c}")

    st.markdown("### Top SHAP drivers:")
    shap_items = eval(row["shap_top3"]) if isinstance(row["shap_top3"], str) else row["shap_top3"]
    if shap_items:
        for s in shap_items:
            st.markdown(f"• {s}")
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard file written: /content/app.py")

# Git: Cell 4 — Streamlit dashboard file written

✅ Dashboard file written: /content/app.py


In [8]:
# Cell 5 — Launch dashboard with localtunnel (no account needed)
import subprocess
import threading
import time

# Install localtunnel
subprocess.run(["npm", "install", "-g", "localtunnel"], capture_output=True)

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to boot
time.sleep(6)

# Start localtunnel
tunnel = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
output = tunnel.stdout.readline().decode("utf-8").strip()

print("=" * 55)
print("👁️  EYE OF RA DASHBOARD IS LIVE")
print("=" * 55)
print(f"\n🌐 Open this URL in your browser:")
print(f"   {output}")
print(f"\nIf it asks for a password, go to https://loca.lt/mytunnelpassword")
print(f"Copy the password shown there and paste it into the tunnel page.")
print(f"\nKeep this cell running — closing it kills the dashboard.")

# Git: Cell 5 — dashboard launched via localtunnel

👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL in your browser:
   your url is: https://calm-dingos-taste.loca.lt

If it asks for a password, go to https://loca.lt/mytunnelpassword
Copy the password shown there and paste it into the tunnel page.

Keep this cell running — closing it kills the dashboard.


In [9]:
# Cell 5 — Launch dashboard with cloudflared
import subprocess
import threading
import time

# Install cloudflared
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("✅ cloudflared installed")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to boot
time.sleep(6)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract the URL from output
print("⏳ Starting tunnel...")
for _ in range(20):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        url = [x for x in line.split() if "trycloudflare.com" in x][0]
        print("=" * 55)
        print("👁️  EYE OF RA DASHBOARD IS LIVE")
        print("=" * 55)
        print(f"\n🌐 Open this URL:")
        print(f"   {url}")
        print("\nNo password needed.")
        break
    time.sleep(1)

# Git: Cell 5 — dashboard launched via cloudflared

✅ cloudflared installed
✅ Streamlit running
⏳ Starting tunnel...
👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL:
   trycloudflare.com...

No password needed.


In [10]:
# Cell 5 — Launch dashboard with cloudflared (fixed)
import subprocess
import threading
import time
import shutil

# Confirm streamlit path
streamlit_path = shutil.which("streamlit")
print(f"✅ Streamlit found at: {streamlit_path}")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        streamlit_path, "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(6)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract URL
print("⏳ Starting tunnel...")
for _ in range(30):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        # Extract just the URL
        parts = line.split()
        for part in parts:
            if "trycloudflare.com" in part:
                url = part.strip()
                print("=" * 55)
                print("👁️  EYE OF RA DASHBOARD IS LIVE")
                print("=" * 55)
                print(f"\n🌐 Open this URL:")
                print(f"   {url}")
                print("\nNo password needed.")
                break
        break
    time.sleep(1)

# Git: Cell 5 — dashboard launched, streamlit path fixed

✅ Streamlit found at: /usr/local/bin/streamlit
✅ Streamlit running
⏳ Starting tunnel...
👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL:
   trycloudflare.com...

No password needed.


In [11]:
# Cell 5 — Reinstall Streamlit and launch with cloudflared
import subprocess
import threading
import time

# Reinstall streamlit
print("⏳ Installing Streamlit...")
subprocess.run(["pip", "install", "streamlit", "-q"])

# Confirm path
result = subprocess.run(["which", "streamlit"], capture_output=True, text=True)
streamlit_path = result.stdout.strip()
print(f"✅ Streamlit found at: {streamlit_path}")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        streamlit_path, "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(8)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract URL
print("⏳ Starting tunnel...")
for _ in range(30):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        parts = line.split()
        for part in parts:
            if "trycloudflare.com" in part:
                url = part.strip()
                print("=" * 55)
                print("👁️  EYE OF RA DASHBOARD IS LIVE")
                print("=" * 55)
                print(f"\n🌐 Open this URL:")
                print(f"   {url}")
                print("\nNo password needed.")
                break
        break
    time.sleep(1)

# Git: Cell 5 — streamlit reinstalled, dashboard launched

⏳ Installing Streamlit...
✅ Streamlit found at: /usr/local/bin/streamlit
✅ Streamlit running
⏳ Starting tunnel...
👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL:
   trycloudflare.com...

No password needed.


In [12]:
# Cell 5 — Fix URL extraction
import subprocess
import threading
import time

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "/usr/local/bin/streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(8)
print("✅ Streamlit running")

# Start cloudflared and print ALL stderr lines so we can see the full URL
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Reading tunnel output...")
for _ in range(40):
    line = tunnel.stderr.readline().decode("utf-8").strip()
    if line:
        print(line)  # print every line so we can see exactly what comes out
    if "trycloudflare.com" in line:
        break
    time.sleep(1)

# Git: Cell 5 — debug URL extraction

✅ Streamlit running
⏳ Reading tunnel output...
2026-06-08T15:50:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-08T15:50:22Z INF Requesting new quick Tunnel on trycloudflare.com...


In [18]:
# Cell 5b — Keep reading tunnel output until URL appears
print("⏳ Waiting for URL...")
for _ in range(60):
    line = tunnel.stderr.readline().decode("utf-8").strip()
    if line:
        print(line)
    if "trycloudflare.com" in line and "http" in line:
        break
    time.sleep(1)

⏳ Waiting for URL...
2026-06-08T16:25:19Z INF +--------------------------------------------------------------------------------------------+
2026-06-08T16:25:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-08T16:25:19Z INF |  https://suspended-establish-officially-hence.trycloudflare.com                            |


In [ ]:
# Cell 6 — Standardize entity names in master dataset
# Fix duplicate/inconsistent entity names before re-scoring

entity_mapping = {
    # BOST
    "Bulk Oil Storage and Transportation": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage Transportation Company Ltd. (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage and Transportation company Limited (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage and Transportation Company Ltd. (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",

    # Bank of Ghana
    "Bank of Ghana": "Bank of Ghana (BoG)",

    # Accra Technical University
    "Accra Technical University": "Accra Technical University (ATU)",

    # Ghana Gas
    "Ghana National Gas Company": "Ghana National Gas Company (Ghana Gas)",
    "Ghana Gas": "Ghana National Gas Company (Ghana Gas)",
    "Ghana National Gas Company (GNGC)": "Ghana National Gas Company (Ghana Gas)",

    # GPHA
    "Ghana Ports and Harbours Authority": "Ghana Ports and Harbours Authority (GPHA)",

    # COCOBOD
    "Ghana Cocoa Board": "Ghana Cocoa Board (COCOBOD)",

    # Ministry of Finance
    "Ministry of Finance": "Ministry of Finance (MoF)",

    # Korle Bu
    "Korle-Bu Teaching Hospital": "Korle Bu Teaching Hospital (KBTH)",
    "Korle Bu Teaching Hospital": "Korle Bu Teaching Hospital (KBTH)",
}

# Apply mapping
df['entity'] = df['entity'].replace(entity_mapping)

# Check remaining duplicates
print("=" * 55)
print("👁️  ENTITY NAME STANDARDIZATION")
print("=" * 55)
print(f"Unique entities before: 133")
print(f"Unique entities after:  {df['entity'].nunique()}")
print(f"\nAll unique entity names:")
for e in sorted(df['entity'].unique()):
    print(f"  {e}")

# Git: Cell 6 — entity name standardization

👁️  ENTITY NAME STANDARDIZATION
Unique entities before: 133
Unique entities after:  123

All unique entity names:
  Accra Technical University (ATU)
  Bank of Ghana (BoG)
  Bulk Oil Storage and Transportatio n Company Limited (BOST)
  Bulk Oil Storage and Transportation Company Limited (BOST)
  Cape Coast Technical University (CCTU)
  Centre for Plant Medicine Research
  Coastal Development Authority (CODA)
  Controller and Accountant General’s Department (CAGD)
  Driver and Vehicle Licensing Authority
  Driver and Vehicle Licensing Authority (DVLA)
  Electoral Commission
  Electoral Commission (EC)
  GIFEC
  GNPC
  Ghana Airport Company Ltd
  Ghana Airport Company Ltd. (GACL)
  Ghana Airports Company Limited (GACL)
  Ghana Cocoa Board (COCOBOD)
  Ghana Cocoa Board (Cocoa Board)
  Ghana College of Physicians and Surgeons
  Ghana Commodity Exchange (GCX)
  Ghana Education Services (GES)
  Ghana Export Promotion Authority (GEPA)
  Ghana Export Promotion Authority (GEPA))
  Ghana Geologic

In [ ]:
# Cell 7 — Fix all remaining entity name duplicates

entity_mapping_2 = {
    # BOST (typo with space)
    "Bulk Oil Storage and Transportatio n Company Limited (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",

    # DVLA
    "Driver and Vehicle Licensing Authority": "Driver and Vehicle Licensing Authority (DVLA)",

    # Electoral Commission
    "Electoral Commission": "Electoral Commission (EC)",

    # GIFEC
    "GIFEC": "Ghana Investment Fund for Electronic Communications (GIFEC)",

    # Ghana Airport
    "Ghana Airport Company Ltd": "Ghana Airports Company Limited (GACL)",
    "Ghana Airport Company Ltd. (GACL)": "Ghana Airports Company Limited (GACL)",

    # Ghana Cocoa Board
    "Ghana Cocoa Board (Cocoa Board)": "Ghana Cocoa Board (COCOBOD)",

    # Ghana Export Promotion Authority (typo extra bracket)
    "Ghana Export Promotion Authority (GEPA))": "Ghana Export Promotion Authority (GEPA)",

    # Ghana Gas / GNPC confusion
    "Ghana National Gas Company (GNPC)": "Ghana National Gas Company (Ghana Gas)",
    "Ghana National Gas Company Ltd": "Ghana National Gas Company (Ghana Gas)",
    "GNPC": "Ghana National Petroleum Corporation (GNPC)",

    # Ghana Maritime Authority
    "Ghana Maritime Authority": "Ghana Maritime Authority (GMA)",

    # Ghana Railway
    "Ghana Railway Authority": "Ghana Railways Development Authority",
    "Ghana Railway Company Ltd": "Ghana Railways Development Authority",
    "Ghana Railways Company Ltd": "Ghana Railways Development Authority",
    "Ministry of Railways Development": "Ministry of Railway Development (MoRD)",

    # Ghana Statistical Service
    "Ghana Statistical Service": "Ghana Statistical Service (GSS)",

    # Ministry of Education
    "Ministry of Education": "Ministry of Education (MoE)",

    # Ministry of Health
    "Ministry of Health": "Ministry of Health (MoH)",
    "Ministry of Health (MOH)": "Ministry of Health (MoH)",

    # Ministry of Local Government
    "Ministry of Local Government Decentralization and Rural Development (MLGDRD)": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",
    "Ministry of Local Government and Rural Development (MLRD)": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",
    "Ministry of Local Government, Decentralization and Rural Development (MLGDRD),": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",

    # Ministry of Youth and Sports
    "Ministry of Youth and Sports (MoYS)": "Ministry of Youth and Sports (MOYS)",

    # NHIA
    "NHIA": "National Health Insurance Authority (NHIA)",
    "National Health Insurance Authority": "National Health Insurance Authority (NHIA)",
    "National Health Insurance Authority (NHIS)": "National Health Insurance Authority (NHIA)",

    # National Cardiothoracic Centre (typo with space)
    "National Cardiothoraci c Centre (NCTC)": "National Cardiothoracic Centre (NCTC)",

    # National Centre for Radiotherapy
    "National Centre for Radiotherapy and": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",
    "Nuclear Medicine (NCRNM) of Korle Bu Teaching Hospital (KBTH)": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",
    "National Centre for Radiotherapy and Nuclear Medicine (NCRNM) of Korle Bu Teaching Hospital (KBTH)": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",

    # National Communications Authority (typo with space)
    "National Communicati ons Authority (NCA)": "National Communications Authority (NCA)",

    # National Petroleum Authority
    "National Petroleum Authority": "National Petroleum Authority (NPA)",

    # Office of Attorney General (typo with space)
    "Office of the Attorney- General and Ministry of Justice": "Office of the Attorney-General and Ministry of Justice",

    # PSC Tema Shipyard
    "PSC Tema Shipyard": "PSC Tema Shipyard Limited (Tema Shipyard)",
    "PSC Tema Shipyard Limited": "PSC Tema Shipyard Limited (Tema Shipyard)",

    # Scholarship Secretariat
    "Scholarship Secretariat": "Scholarship Secretariat (SLA)",

    # Volta River Authority
    "Volta River Authority": "Volta River Authority (VRA)",
}

df['entity'] = df['entity'].replace(entity_mapping_2)

print("=" * 55)
print("👁️  ENTITY NAMES — FINAL CLEAN")
print("=" * 55)
print(f"Unique entities now: {df['entity'].nunique()}")
print(f"\nAll unique entity names:")
for e in sorted(df['entity'].unique()):
    print(f"  {e}")

# Git: "Cell 7 — all entity name duplicates fixed"

👁️  ENTITY NAMES — FINAL CLEAN
Unique entities now: 87

All unique entity names:
  Accra Technical University (ATU)
  Bank of Ghana (BoG)
  Bulk Oil Storage and Transportation Company Limited (BOST)
  Cape Coast Technical University (CCTU)
  Centre for Plant Medicine Research
  Coastal Development Authority (CODA)
  Controller and Accountant General’s Department (CAGD)
  Driver and Vehicle Licensing Authority (DVLA)
  Electoral Commission (EC)
  Ghana Airports Company Limited (GACL)
  Ghana Cocoa Board (COCOBOD)
  Ghana College of Physicians and Surgeons
  Ghana Commodity Exchange (GCX)
  Ghana Education Services (GES)
  Ghana Export Promotion Authority (GEPA)
  Ghana Geological Survey Authority (GGSA)
  Ghana Health Services (GHS)
  Ghana Investment Fund for Electronic Communications (GIFEC)
  Ghana Maritime Authority (GMA)
  Ghana National Gas Company (Ghana Gas)
  Ghana National Petroleum Corporation (GNPC)
  Ghana Police Service (GPS)
  Ghana Ports and Harbours Authority (GPHA)
  G

In [ ]:
# Cell 8 — Save clean dataset and re-run full scoring pipeline

# Save clean master dataset
df.to_csv('/content/eye_of_ra_master_dataset_v4.csv', index=False)
print("✅ Clean dataset saved: eye_of_ra_master_dataset_v4.csv")

# Re-run all 5 engines on clean data
# ── ENGINE 1 ──
def bid_manipulation_score(row, df):
    score = 0
    flags = []
    if row['is_sole_source'] == 1:
        score += 30
        flags.append("Sole source procurement used (S.38-41 Act 663)")
    if row['is_variation'] == 1 and row['audit_flagged'] == 1:
        score += 25
        flags.append("Variation issued on audit-flagged contract (S.59 Act 663)")
    if row['award_concentration'] > 0.3:
        score += 20
        flags.append(f"Supplier holds {row['award_concentration']*100:.1f}% of entity awards — concentration risk")
    if row['method_abuse_score'] > 0.5:
        score += 15
        flags.append("Procurement method abuse pattern detected (S.40-41 Act 663)")
    if row['variation_abuse_rate'] > 0.3:
        score += 10
        flags.append(f"Entity variation abuse rate: {row['variation_abuse_rate']*100:.1f}%")
    return min(score, 100), flags

# ── ENGINE 2 ──
def build_supplier_profiles(df):
    profiles = {}
    for supplier in df['supplier'].unique():
        s = df[df['supplier'] == supplier]
        total = len(s)
        score = 0
        flags = []
        if s['fraud_label'].sum() > 0:
            score += 40
            flags.append(f"Supplier linked to {int(s['fraud_label'].sum())} confirmed fraud case(s)")
        if s['audit_flagged'].sum() > 0:
            score += 20
            flags.append(f"Supplier appears in {int(s['audit_flagged'].sum())} Auditor-General finding(s)")
        ss_rate = s['is_sole_source'].sum() / total if total > 0 else 0
        if ss_rate > 0.5:
            score += 20
            flags.append(f"{int(ss_rate*100)}% of awards were sole source — bypasses competition (S.38-41)")
        var_rate = s['is_variation'].sum() / total if total > 0 else 0
        if var_rate > 0.4:
            score += 15
            flags.append(f"{int(var_rate*100)}% of contracts have variations — inflation risk")
        if total >= 3 and s['entity'].nunique() == 1:
            score += 5
            flags.append(f"All {total} contracts with a single entity — capture risk")
        profiles[supplier] = {'engine2_score': min(score, 100), 'engine2_flags': flags}
    return profiles

# ── ENGINE 3 ──
def build_entity_variation_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        var_rate = e['is_variation'].sum() / total if total > 0 else 0
        audit_flagged_var = e[(e['is_variation']==1) & (e['audit_flagged']==1)].shape[0]
        fraud_var = e[(e['is_variation']==1) & (e['fraud_label']==1)].shape[0]
        score = 0
        flags = []
        if var_rate > 0.4:
            score += 35
            flags.append(f"{int(var_rate*100)}% of contracts are variations — systematic inflation pattern (S.87 Act 663)")
        elif var_rate > 0.2:
            score += 15
            flags.append(f"{int(var_rate*100)}% variation rate — elevated above normal threshold")
        if audit_flagged_var > 0:
            score += 30
            flags.append(f"{audit_flagged_var} variation(s) independently flagged by Auditor-General")
        if fraud_var > 0:
            score += 35
            flags.append(f"{fraud_var} variation contract(s) confirmed fraudulent")
        profiles[entity] = {'engine3_score': min(score, 100), 'engine3_flags': flags}
    return profiles

# ── ENGINE 4 ──
def build_sole_source_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        ss_rate = e['is_sole_source'].sum() / total if total > 0 else 0
        audit_ss = e[(e['is_sole_source']==1) & (e['audit_flagged']==1)].shape[0]
        fraud_ss = e[(e['is_sole_source']==1) & (e['fraud_label']==1)].shape[0]
        method_abuse = e['method_abuse_score'].mean()
        score = 0
        flags = []
        if ss_rate > 0.5:
            score += 40
            flags.append(f"{int(ss_rate*100)}% of contracts are sole source — systematic bypass of competition (S.40-41 Act 663)")
        elif ss_rate > 0.25:
            score += 20
            flags.append(f"{int(ss_rate*100)}% sole source rate — above acceptable threshold (S.40-41 Act 663)")
        if audit_ss > 0:
            score += 30
            flags.append(f"{audit_ss} sole source contract(s) flagged by Auditor-General")
        if fraud_ss > 0:
            score += 30
            flags.append(f"{fraud_ss} sole source contract(s) confirmed fraudulent")
        if method_abuse > 0.4:
            score += 15
            flags.append(f"Entity method abuse score: {method_abuse:.2f} — pattern of procurement rule circumvention")
        profiles[entity] = {'engine4_score': min(score, 100), 'engine4_flags': flags}
    return profiles

# ── ENGINE 5 ──
def build_entity_risk_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        fraud_count = e['fraud_label'].sum()
        fraud_rate = fraud_count / total if total > 0 else 0
        audit_rate = e['audit_flagged'].sum() / total if total > 0 else 0
        avg_supplier_risk = e['supplier_risk_score'].mean()
        repeat_supplier_rate = e['repeat_supplier'].mean()
        avg_audit_intensity = e['audit_intensity'].mean()
        score = 0
        flags = []
        if fraud_count > 0:
            score += 35
            flags.append(f"{int(fraud_count)} of {total} contracts confirmed fraudulent ({int(fraud_rate*100)}% fraud rate)")
        if audit_rate > 0.3:
            score += 25
            flags.append(f"{int(audit_rate*100)}% of contracts flagged by Auditor-General — systemic oversight failure (S.2 Act 663)")
        elif e['audit_flagged'].sum() > 0:
            score += 10
            flags.append(f"{int(e['audit_flagged'].sum())} contract(s) flagged by Auditor-General")
        if avg_supplier_risk > 0.6:
            score += 20
            flags.append(f"Average supplier risk score: {avg_supplier_risk:.2f} — entity consistently awards to high-risk suppliers")
        if repeat_supplier_rate > 0.7:
            score += 15
            flags.append(f"{int(repeat_supplier_rate*100)}% repeat supplier rate — limited supplier diversity (S.3 Act 663)")
        if avg_audit_intensity > 0.5:
            score += 5
            flags.append(f"Audit intensity score: {avg_audit_intensity:.2f} — entity under sustained scrutiny")
        profiles[entity] = {'engine5_score': min(score, 100), 'engine5_flags': flags}
    return profiles

# ── RUN ALL ENGINES ──
print("⏳ Running all 5 engines on clean data...")

e1 = df.apply(lambda row: bid_manipulation_score(row, df), axis=1)
df['engine1_score'] = e1.apply(lambda x: x[0])
df['engine1_flags'] = e1.apply(lambda x: x[1])

sp = build_supplier_profiles(df)
df['engine2_score'] = df['supplier'].map(lambda s: sp[s]['engine2_score'])
df['engine2_flags'] = df['supplier'].map(lambda s: sp[s]['engine2_flags'])

evp = build_entity_variation_profiles(df)
df['engine3_score'] = df['entity'].map(lambda e: evp[e]['engine3_score'])
df['engine3_flags'] = df['entity'].map(lambda e: evp[e]['engine3_flags'])

esp = build_sole_source_profiles(df)
df['engine4_score'] = df['entity'].map(lambda e: esp[e]['engine4_score'])
df['engine4_flags'] = df['entity'].map(lambda e: esp[e]['engine4_flags'])

erp = build_entity_risk_profiles(df)
df['engine5_score'] = df['entity'].map(lambda e: erp[e]['engine5_score'])
df['engine5_flags'] = df['entity'].map(lambda e: erp[e]['engine5_flags'])

print("✅ All 5 engines complete")

# ── COMPOSITE SCORE ──
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')
df['ml_score'] = model.predict_proba(df[feature_cols])[:, 1] * 100
df['composite_score'] = (
    df['ml_score'] * 0.40 +
    df['engine1_score'] * 0.15 +
    df['engine2_score'] * 0.15 +
    df['engine3_score'] * 0.10 +
    df['engine4_score'] * 0.10 +
    df['engine5_score'] * 0.10
).round(1).clip(upper=100)

def assign_tier(score):
    if score >= 50:
        return "🔴 ESCALATE"
    elif score >= 35:
        return "🟡 REVIEW"
    else:
        return "🟢 MONITOR"

df['tier'] = df['composite_score'].apply(assign_tier)

# ── SUMMARY ──
fraud = df[df['fraud_label'] == 1]
escalate = fraud[fraud['tier'] == '🔴 ESCALATE']
review = fraud[fraud['tier'] == '🟡 REVIEW']
monitor = fraud[fraud['tier'] == '🟢 MONITOR']

print("=" * 55)
print("👁️  EYE OF RA — CLEAN DATA RESCORE")
print("=" * 55)
print(f"Contracts scored:      {len(df)}")
print(f"Unique entities:       {df['entity'].nunique()}")
print(f"\nFraud Detection:")
print(f"  🔴 ESCALATE: {len(escalate)} of 23 ({len(escalate)/23*100:.0f}%)")
print(f"  🟡 REVIEW:   {len(review)} of 23 ({len(review)/23*100:.0f}%)")
print(f"  🟢 MONITOR:  {len(monitor)} of 23 — missed")
print(f"  Total caught: {len(escalate)+len(review)} of 23 ({(len(escalate)+len(review))/23*100:.0f}%)")
print(f"\nFalse positives in ESCALATE: {len(df[(df['fraud_label']==0) & (df['tier']=='🔴 ESCALATE')])}")

# Save
df.to_csv('/content/eye_of_ra_scored_contracts_v2.csv', index=False)
print(f"\n✅ Saved: eye_of_ra_scored_contracts_v2.csv")

# Git: Cell 8 — clean data rescored, v2 saved

✅ Clean dataset saved: eye_of_ra_master_dataset_v4.csv
⏳ Running all 5 engines on clean data...
✅ All 5 engines complete
👁️  EYE OF RA — CLEAN DATA RESCORE
Contracts scored:      262
Unique entities:       87

Fraud Detection:
  🔴 ESCALATE: 18 of 23 (78%)
  🟡 REVIEW:   2 of 23 (9%)
  🟢 MONITOR:  3 of 23 — missed
  Total caught: 20 of 23 (87%)

False positives in ESCALATE: 0

✅ Saved: eye_of_ra_scored_contracts_v2.csv


In [ ]:
# Cell 9 — Update dashboard to use clean v2 files

dashboard_code = open("/content/app.py").read()
dashboard_code = dashboard_code.replace(
    "eye_of_ra_scored_contracts_v1.csv",
    "eye_of_ra_scored_contracts_v2.csv"
)

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard updated to use scored_contracts_v2.csv")
print("⏳ Restart the Streamlit server by rerunning Cell 5 to see changes.")

# Git: Cell 9 — dashboard updated to v2 scored contracts

✅ Dashboard updated to use scored_contracts_v2.csv
⏳ Restart the Streamlit server by rerunning Cell 5 to see changes.


In [ ]:
# Cell 10 — Save complete scored contracts with all columns dashboard needs

scored_v2 = df[[
    'entity',
    'supplier',
    'fraud_label',
    'composite_score',
    'tier',
    'ml_score',
    'engine1_score',
    'engine2_score',
    'engine3_score',
    'engine4_score',
    'engine5_score',
    'engine1_flags',
    'engine2_flags',
    'engine3_flags',
    'engine4_flags',
    'engine5_flags',
]].copy()

# Combine all flags into one column
scored_v2['flags'] = scored_v2.apply(
    lambda row: (
        row['engine1_flags'] +
        row['engine2_flags'] +
        row['engine3_flags'] +
        row['engine4_flags'] +
        row['engine5_flags']
    ), axis=1
)

# Add legal citations
def get_legal_citations(row):
    refs = set()
    if row['is_sole_source'] == 1:
        refs.add("S.38-41 Act 663 (Sole Source Procurement)")
    if row['is_variation'] == 1:
        refs.add("S.87 Act 663 (Contract Variations)")
    if row['audit_flagged'] == 1:
        refs.add("S.92-93 Act 663 (Procurement Offences)")
    if row['method_abuse_score'] > 0.4:
        refs.add("S.40-41 Act 663 (Procurement Method Abuse)")
    if row['award_concentration'] > 0.3:
        refs.add("S.3 Act 663 (Transparency and Accountability)")
    return list(refs)

scored_v2['legal_citations'] = df.apply(get_legal_citations, axis=1)

# Add SHAP explanations
import shap
explainer = shap.TreeExplainer(model)
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')

def get_shap_top3(idx):
    row = df.loc[idx]
    features = row[feature_cols].values.reshape(1, -1)
    shap_values = explainer.shap_values(features)
    shap_pairs = list(zip(feature_cols, shap_values[0]))
    shap_sorted = sorted(shap_pairs, key=lambda x: abs(x[1]), reverse=True)[:3]
    result = []
    for feat, val in shap_sorted:
        direction = "increases" if val > 0 else "decreases"
        result.append(f"{feat} {direction} fraud risk (impact: {abs(val):.3f})")
    return result

print("⏳ Computing SHAP values for all contracts...")
scored_v2['shap_top3'] = [get_shap_top3(idx) for idx in df.index]

# Drop engine-specific flag columns — dashboard only needs combined flags
scored_v2 = scored_v2.drop(columns=[
    'engine1_flags', 'engine2_flags', 'engine3_flags',
    'engine4_flags', 'engine5_flags'
])

scored_v2.to_csv('/content/eye_of_ra_scored_contracts_v2.csv', index=False)

print(f"✅ Saved: eye_of_ra_scored_contracts_v2.csv")
print(f"Columns: {list(scored_v2.columns)}")
print(f"Rows: {len(scored_v2)}")

# Git: Cell 10 — complete scored contracts v2 saved with all dashboard columns

⏳ Computing SHAP values for all contracts...
✅ Saved: eye_of_ra_scored_contracts_v2.csv
Columns: ['entity', 'supplier', 'fraud_label', 'composite_score', 'tier', 'ml_score', 'engine1_score', 'engine2_score', 'engine3_score', 'engine4_score', 'engine5_score', 'flags', 'legal_citations', 'shap_top3']
Rows: 262


In [ ]:
# Cell 11 — Rewrite app.py to match v2 CSV structure

dashboard_code = '''
import streamlit as st
import pandas as pd
import ast
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Eye of Ra 👁️",
    page_icon="👁️",
    layout="wide"
)

@st.cache_data
def load_data():
    scored = pd.read_csv("/content/eye_of_ra_scored_contracts_v2.csv")
    return scored

scored = load_data()

def parse_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return []

# ── SIDEBAR ──
st.sidebar.image("https://flagcdn.com/w80/gh.png", width=60)
st.sidebar.title("👁️ Eye of Ra")
st.sidebar.markdown("AI Procurement Fraud Detection for Ghana")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])
st.sidebar.markdown("---")
st.sidebar.markdown("Built on Ghana PPA & Auditor-General data")
st.sidebar.markdown("Grounded in Public Procurement Act 663")

# ── PAGE 1: OVERVIEW ──
if page == "📊 Overview":
    st.title("👁️ Eye of Ra — Procurement Fraud Detection")
    st.markdown("### Ghana AI Summit 2026 | Real Data. Real Flags. Real Accountability.")
    st.markdown("---")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col3.metric("🔴 Escalate", str(len(scored[scored["tier"] == "🔴 ESCALATE"])))
    col4.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("---")
    col1, col2 = st.columns(2)

    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#d32f2f",
                "🟡 REVIEW": "#f9a825",
                "🟢 MONITOR": "#388e3c"
            }
        )
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        fraud = scored[scored["fraud_label"] == 1]
        escalate_n = len(fraud[fraud["tier"] == "🔴 ESCALATE"])
        review_n = len(fraud[fraud["tier"] == "🟡 REVIEW"])
        monitor_n = len(fraud[fraud["tier"] == "🟢 MONITOR"])
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\\n(caught)", "🟡 REVIEW\\n(caught)", "🟢 MONITOR\\n(missed)"],
            y=[escalate_n, review_n, monitor_n],
            marker_color=["#d32f2f", "#f9a825", "#388e3c"],
            text=[escalate_n, review_n, monitor_n],
            textposition="outside"
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            yaxis_title="Number of Contracts",
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    st.markdown("### Score Distribution — Fraud vs Clean")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#388e3c", 1: "#d32f2f"},
        labels={"fraud_label": "Fraud", "composite_score": "Composite Risk Score"},
        title="Risk Score Distribution"
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#d32f2f", annotation_text="ESCALATE threshold")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f9a825", annotation_text="REVIEW threshold")
    st.plotly_chart(fig3, use_container_width=True)

# ── PAGE 2: FLAGGED CONTRACTS ──
elif page == "🔴 Flagged Contracts":
    st.title("🔴 Flagged Contracts")
    st.markdown("Contracts in ESCALATE and REVIEW tiers — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False)
    st.markdown(f"**{len(filtered)} contracts shown**")
    st.dataframe(
        filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 3: ENTITY SCORECARDS ──
elif page == "🏛️ Entity Scorecards":
    st.title("🏛️ Entity Integrity Scorecards")
    st.markdown("Search any procurement entity and see their full risk profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Select an entity:", entity_list)

    entity_contracts = scored[scored["entity"] == selected_entity]
    max_score = entity_contracts["composite_score"].max()
    fraud_count = entity_contracts["fraud_label"].sum()
    total = len(entity_contracts)
    escalate_count = len(entity_contracts[entity_contracts["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Contracts", total)
    col2.metric("Max Risk Score", f"{max_score:.1f}")
    col3.metric("Confirmed Fraud", int(fraud_count))
    col4.metric("Escalated", escalate_count)

    st.markdown("---")
    st.markdown("### All contracts for this entity:")
    st.dataframe(
        entity_contracts[["supplier", "composite_score", "tier", "fraud_label"]]
        .sort_values("composite_score", ascending=False)
        .reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 4: CONTRACT DEEP DIVE ──
elif page == "🔍 Contract Deep Dive":
    st.title("🔍 Contract Deep Dive")
    st.markdown("Select any contract for a full risk breakdown.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:50]} | Score: {scored.loc[i, 'composite_score']}"
    )

    row = scored.loc[contract_idx]

    col1, col2 = st.columns(2)
    with col1:
        st.markdown(f"**Entity:** {row['entity']}")
        st.markdown(f"**Supplier:** {row['supplier']}")
        st.markdown(f"**Composite Score:** {row['composite_score']}")
        st.markdown(f"**Tier:** {row['tier']}")
        st.markdown(f"**Fraud Confirmed:** {'✅ Yes' if row['fraud_label'] == 1 else '⬜ No'}")

    with col2:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=row["composite_score"],
            title={"text": "Risk Score"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#d32f2f" if row["composite_score"] >= 50 else "#f9a825" if row["composite_score"] >= 35 else "#388e3c"},
                "steps": [
                    {"range": [0, 35], "color": "#e8f5e9"},
                    {"range": [35, 50], "color": "#fff9c4"},
                    {"range": [50, 100], "color": "#ffebee"}
                ],
                "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.75, "value": row["composite_score"]}
            }
        ))
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("### Flags raised:")
    flags = parse_list(row["flags"])
    if flags:
        for f in flags:
            st.markdown(f"• {f}")
    else:
        st.markdown("No flags raised.")

    st.markdown("### Legal citations:")
    citations = parse_list(row["legal_citations"])
    if citations:
        for c in citations:
            st.markdown(f"• {c}")
    else:
        st.markdown("No citations.")

    st.markdown("### Top SHAP drivers:")
    shap_items = parse_list(row["shap_top3"])
    if shap_items:
        for s in shap_items:
            st.markdown(f"• {s}")
    else:
        st.markdown("No SHAP data.")
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ app.py rewritten — now compatible with v2 CSV")
print("⏳ Rerun Cell 5 to relaunch the dashboard.")

# Git: Cell 11 — app.py rewritten for v2 CSV compatibility

✅ app.py rewritten — now compatible with v2 CSV
⏳ Rerun Cell 5 to relaunch the dashboard.


In [ ]:
# Cell 12 — Check scored v2 columns and fix flags
import pandas as pd
scored = pd.read_csv('/content/eye_of_ra_scored_contracts_v2.csv')
print("Columns in scored v2:")
print(list(scored.columns))
print(f"\nSample flags value:")
print(scored['flags'].iloc[0])

# Git: Cell 12 — debug flags column

Columns in scored v2:
['entity', 'supplier', 'fraud_label', 'composite_score', 'tier', 'ml_score', 'engine1_score', 'engine2_score', 'engine3_score', 'engine4_score', 'engine5_score', 'flags', 'legal_citations', 'shap_top3']

Sample flags value:
['Supplier appears in 1 Auditor-General finding(s)', '1 variation(s) independently flagged by Auditor-General', '1 sole source contract(s) flagged by Auditor-General', '2 of 13 contracts confirmed fraudulent (15% fraud rate)', '84% of contracts flagged by Auditor-General — systemic oversight failure (S.2 Act 663)', 'Audit intensity score: 15.23 — entity under sustained scrutiny']


In [ ]:
# Cell 12 — Clear Streamlit cache and update app to force reload

dashboard_fix = '''
import streamlit as st
import pandas as pd
import ast
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Eye of Ra 👁️",
    page_icon="👁️",
    layout="wide"
)

def load_data():
    scored = pd.read_csv("/content/eye_of_ra_scored_contracts_v2.csv")
    return scored

scored = load_data()

def parse_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(str(val))
    except:
        return []

# ── SIDEBAR ──
st.sidebar.image("https://flagcdn.com/w80/gh.png", width=60)
st.sidebar.title("👁️ Eye of Ra")
st.sidebar.markdown("AI Procurement Fraud Detection for Ghana")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])
st.sidebar.markdown("---")
st.sidebar.markdown("Built on Ghana PPA & Auditor-General data")
st.sidebar.markdown("Grounded in Public Procurement Act 663")

# ── PAGE 1: OVERVIEW ──
if page == "📊 Overview":
    st.title("👁️ Eye of Ra — Procurement Fraud Detection")
    st.markdown("### Ghana AI Summit 2026 | Real Data. Real Flags. Real Accountability.")
    st.markdown("---")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col3.metric("🔴 Escalate", str(len(scored[scored["tier"] == "🔴 ESCALATE"])))
    col4.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("---")
    col1, col2 = st.columns(2)

    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#d32f2f",
                "🟡 REVIEW": "#f9a825",
                "🟢 MONITOR": "#388e3c"
            }
        )
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        fraud = scored[scored["fraud_label"] == 1]
        escalate_n = len(fraud[fraud["tier"] == "🔴 ESCALATE"])
        review_n = len(fraud[fraud["tier"] == "🟡 REVIEW"])
        monitor_n = len(fraud[fraud["tier"] == "🟢 MONITOR"])
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\\n(caught)", "🟡 REVIEW\\n(caught)", "🟢 MONITOR\\n(missed)"],
            y=[escalate_n, review_n, monitor_n],
            marker_color=["#d32f2f", "#f9a825", "#388e3c"],
            text=[escalate_n, review_n, monitor_n],
            textposition="outside"
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            yaxis_title="Number of Contracts",
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    st.markdown("### Score Distribution — Fraud vs Clean")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#388e3c", 1: "#d32f2f"},
        labels={"fraud_label": "Fraud", "composite_score": "Composite Risk Score"},
        title="Risk Score Distribution"
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#d32f2f", annotation_text="ESCALATE threshold")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f9a825", annotation_text="REVIEW threshold")
    st.plotly_chart(fig3, use_container_width=True)

# ── PAGE 2: FLAGGED CONTRACTS ──
elif page == "🔴 Flagged Contracts":
    st.title("🔴 Flagged Contracts")
    st.markdown("Contracts in ESCALATE and REVIEW tiers — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False)
    st.markdown(f"**{len(filtered)} contracts shown**")
    st.dataframe(
        filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 3: ENTITY SCORECARDS ──
elif page == "🏛️ Entity Scorecards":
    st.title("🏛️ Entity Integrity Scorecards")
    st.markdown("Search any procurement entity and see their full risk profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Select an entity:", entity_list)

    entity_contracts = scored[scored["entity"] == selected_entity]
    max_score = entity_contracts["composite_score"].max()
    fraud_count = entity_contracts["fraud_label"].sum()
    total = len(entity_contracts)
    escalate_count = len(entity_contracts[entity_contracts["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Contracts", total)
    col2.metric("Max Risk Score", f"{max_score:.1f}")
    col3.metric("Confirmed Fraud", int(fraud_count))
    col4.metric("Escalated", escalate_count)

    st.markdown("---")
    st.markdown("### All contracts for this entity:")
    st.dataframe(
        entity_contracts[["supplier", "composite_score", "tier", "fraud_label"]]
        .sort_values("composite_score", ascending=False)
        .reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 4: CONTRACT DEEP DIVE ──
elif page == "🔍 Contract Deep Dive":
    st.title("🔍 Contract Deep Dive")
    st.markdown("Select any contract for a full risk breakdown.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:50]} | Score: {scored.loc[i, 'composite_score']}"
    )

    row = scored.loc[contract_idx]

    col1, col2 = st.columns(2)
    with col1:
        st.markdown(f"**Entity:** {row['entity']}")
        st.markdown(f"**Supplier:** {row['supplier']}")
        st.markdown(f"**Composite Score:** {row['composite_score']}")
        st.markdown(f"**Tier:** {row['tier']}")
        st.markdown(f"**Fraud Confirmed:** {'✅ Yes' if row['fraud_label'] == 1 else '⬜ No'}")

    with col2:
        score_val = float(row["composite_score"])
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=score_val,
            title={"text": "Risk Score"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#d32f2f" if score_val >= 50 else "#f9a825" if score_val >= 35 else "#388e3c"},
                "steps": [
                    {"range": [0, 35], "color": "#e8f5e9"},
                    {"range": [35, 50], "color": "#fff9c4"},
                    {"range": [50, 100], "color": "#ffebee"}
                ],
                "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.75, "value": score_val}
            }
        ))
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")

    st.markdown("### Flags raised:")
    flags = parse_list(row["flags"])
    if flags:
        for f in flags:
            st.markdown(f"• {f}")
    else:
        st.markdown("No flags raised.")

    st.markdown("### Legal citations:")
    citations = parse_list(row["legal_citations"])
    if citations:
        for c in citations:
            st.markdown(f"• {c}")
    else:
        st.markdown("No citations.")

    st.markdown("### Top SHAP drivers:")
    shap_items = parse_list(row["shap_top3"])
    if shap_items:
        for s in shap_items:
            st.markdown(f"• {s}")
    else:
        st.markdown("No SHAP data.")
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_fix)

print("✅ app.py rewritten — cache removed, flags fixed")
print("⏳ Rerun Cell 5 to relaunch.")

# Git: Cell 12 — app.py cache removed, flags fixed

✅ app.py rewritten — cache removed, flags fixed
⏳ Rerun Cell 5 to relaunch.


In [4]:
# Cell 13 — Full dashboard rewrite with Ghana-identity design

dashboard_code = '''
import streamlit as st
import pandas as pd
import ast
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

st.set_page_config(
    page_title="Eye of Ra 👁️ — Ghana Procurement Intelligence",
    page_icon="👁️",
    layout="wide"
)

# ── GLOBAL STYLES ──
st.markdown("""
<style>
    /* Base */
    .stApp {
        background-color: #1a1a1a;
        color: #f0ece2;
    }

    /* Sidebar */
    [data-testid="stSidebar"] {
        background-color: #111111;
        border-right: 2px solid #f5a623;
    }

    /* Sidebar text */
    [data-testid="stSidebar"] * {
        color: #f0ece2 !important;
    }

    /* Metric cards */
    [data-testid="metric-container"] {
        background-color: #242424;
        border-left: 4px solid #f5a623;
        border-radius: 6px;
        padding: 16px;
    }
    [data-testid="metric-container"] label {
        color: #f5a623 !important;
        font-size: 0.75rem !important;
        text-transform: uppercase;
        letter-spacing: 1px;
    }
    [data-testid="metric-container"] [data-testid="stMetricValue"] {
        color: #f0ece2 !important;
        font-size: 2rem !important;
        font-weight: 800 !important;
    }
    [data-testid="metric-container"] [data-testid="stMetricDelta"] {
        color: #2d6a4f !important;
    }

    /* Headers */
    h1 { color: #f5a623 !important; font-weight: 900 !important; letter-spacing: -1px; }
    h2 { color: #f0ece2 !important; font-weight: 700 !important; }
    h3 { color: #f5a623 !important; font-weight: 700 !important; }

    /* Divider */
    hr { border-color: #f5a623 !important; opacity: 0.3; }

    /* Dataframe */
    [data-testid="stDataFrame"] {
        border: 1px solid #333333;
        border-radius: 6px;
    }

    /* Flag card */
    .flag-card {
        background-color: #2a1a1a;
        border-left: 4px solid #c0392b;
        border-radius: 6px;
        padding: 12px 16px;
        margin: 6px 0;
        color: #f0ece2;
        font-size: 0.9rem;
    }
    .review-card {
        background-color: #2a2410;
        border-left: 4px solid #f5a623;
        border-radius: 6px;
        padding: 12px 16px;
        margin: 6px 0;
        color: #f0ece2;
        font-size: 0.9rem;
    }
    .citation-card {
        background-color: #1a2a1a;
        border-left: 4px solid #2d6a4f;
        border-radius: 6px;
        padding: 12px 16px;
        margin: 6px 0;
        color: #f0ece2;
        font-size: 0.9rem;
    }
    .shap-card {
        background-color: #1e1e2a;
        border-left: 4px solid #7b8cde;
        border-radius: 6px;
        padding: 12px 16px;
        margin: 6px 0;
        color: #f0ece2;
        font-size: 0.9rem;
    }

    /* Hero banner */
    .hero-banner {
        background: linear-gradient(135deg, #242424 0%, #1a1a1a 100%);
        border: 1px solid #f5a623;
        border-radius: 10px;
        padding: 24px 32px;
        margin-bottom: 24px;
    }
    .hero-title {
        font-size: 2.2rem;
        font-weight: 900;
        color: #f5a623;
        margin: 0;
        letter-spacing: -1px;
    }
    .hero-sub {
        font-size: 1rem;
        color: #aaaaaa;
        margin: 4px 0 0 0;
    }
    .hero-stat {
        font-size: 1.1rem;
        color: #c0392b;
        font-weight: 700;
        margin: 12px 0 0 0;
    }

    /* Grade badge */
    .grade-f { background:#c0392b; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-d { background:#e67e22; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-c { background:#f5a623; color:#1a1a1a; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-b { background:#2d6a4f; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-a { background:#1a7a4a; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }

    /* Footer */
    .footer {
        text-align: center;
        color: #555555;
        font-size: 0.75rem;
        padding: 32px 0 16px 0;
        border-top: 1px solid #333333;
        margin-top: 48px;
    }

    /* Selectbox */
    [data-testid="stSelectbox"] > div {
        background-color: #242424 !important;
        border: 1px solid #f5a623 !important;
        color: #f0ece2 !important;
    }

    /* Radio buttons */
    [data-testid="stRadio"] label {
        color: #f0ece2 !important;
    }
</style>
""", unsafe_allow_html=True)

# ── DATA ──
def load_data():
    return pd.read_csv("/content/eye_of_ra_scored_contracts_v2.csv")

scored = load_data()

def parse_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(str(val))
    except:
        return []

def get_grade(score):
    if score >= 60: return "F", "grade-f"
    elif score >= 45: return "D", "grade-d"
    elif score >= 30: return "C", "grade-c"
    elif score >= 15: return "B", "grade-b"
    else: return "A", "grade-a"

# ── SIDEBAR ──
st.sidebar.markdown("""
<div style="text-align:center; padding: 16px 0 8px 0;">
    <img src="https://flagcdn.com/w80/gh.png" width="52" style="border-radius:4px;"/>
    <div style="font-size:1.3rem; font-weight:900; color:#f5a623; margin-top:10px;">👁️ Eye of Ra</div>
    <div style="font-size:0.75rem; color:#888888; margin-top:4px;">Ghana Procurement Intelligence</div>
</div>
<hr style="border-color:#f5a623; opacity:0.3;"/>
""", unsafe_allow_html=True)

page = st.sidebar.radio("", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])

st.sidebar.markdown("""
<hr style="border-color:#333; opacity:0.5;"/>
<div style="font-size:0.72rem; color:#666; padding: 8px 0; line-height:1.8;">
    📂 Source: PPA Annual Reports 2021–2024<br/>
    📂 Source: Auditor-General Reports 2021–2023<br/>
    ⚖️ Grounded in Public Procurement Act 663<br/>
    🔓 Open Source — github.com/Joey-Jireh/eye-of-ra
</div>
<div style="font-size:0.7rem; color:#444; text-align:center; padding-top:8px;">
    Ghana AI Summit 2026
</div>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════
# PAGE 1 — OVERVIEW
# ══════════════════════════════════════════
if page == "📊 Overview":

    st.markdown("""
    <div class="hero-banner">
        <div class="hero-title">👁️ Eye of Ra</div>
        <div class="hero-sub">AI-Powered Procurement Fraud Detection for Ghana &nbsp;|&nbsp; Ghana AI Summit 2026</div>
        <div class="hero-stat">⚠️ Ghana lost GH₵18.4 billion to procurement irregularities in 2024 alone. Less than 0.5% was recovered.</div>
    </div>
    """, unsafe_allow_html=True)

    # Metrics
    fraud = scored[scored["fraud_label"] == 1]
    escalate_n = len(fraud[fraud["tier"] == "🔴 ESCALATE"])
    review_n = len(fraud[fraud["tier"] == "🟡 REVIEW"])
    monitor_n = len(fraud[fraud["tier"] == "🟢 MONITOR"])
    total_escalate = len(scored[scored["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4, col5 = st.columns(5)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Entities Monitored", "87")
    col3.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col4.metric("🔴 Escalated", str(total_escalate))
    col5.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("<br/>", unsafe_allow_html=True)

    col1, col2 = st.columns([1, 1])

    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#c0392b",
                "🟡 REVIEW": "#f5a623",
                "🟢 MONITOR": "#2d6a4f"
            },
            hole=0.45
        )
        fig.update_layout(
            paper_bgcolor="#242424",
            plot_bgcolor="#242424",
            font_color="#f0ece2",
            title_font_color="#f5a623",
            legend_font_color="#f0ece2"
        )
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\n(caught)", "🟡 REVIEW\n(caught)", "🟢 MONITOR\n(missed)"],
            y=[escalate_n, review_n, monitor_n],
            marker_color=["#c0392b", "#f5a623", "#2d6a4f"],
            text=[f"{escalate_n}", f"{review_n}", f"{monitor_n}"],
            textposition="outside",
            textfont=dict(color="#f0ece2", size=16, family="Arial Black")
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            title_font_color="#f5a623",
            paper_bgcolor="#242424",
            plot_bgcolor="#242424",
            font_color="#f0ece2",
            yaxis_title="Contracts",
            yaxis=dict(gridcolor="#333333"),
            xaxis=dict(gridcolor="#333333"),
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    st.markdown("### Score Distribution — Fraud vs Clean Contracts")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#2d6a4f", 1: "#c0392b"},
        labels={"fraud_label": "Contract Type", "composite_score": "Composite Risk Score"},
        title="Clean contracts cluster below 35. Fraud contracts cluster above 50."
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#c0392b",
                   annotation_text="ESCALATE threshold (50)",
                   annotation_font_color="#c0392b")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f5a623",
                   annotation_text="REVIEW threshold (35)",
                   annotation_font_color="#f5a623")
    fig3.update_layout(
        paper_bgcolor="#242424",
        plot_bgcolor="#242424",
        font_color="#f0ece2",
        title_font_color="#f5a623",
        yaxis=dict(gridcolor="#333333"),
        xaxis=dict(gridcolor="#333333")
    )
    st.plotly_chart(fig3, use_container_width=True)

    # Top 10 entities
    st.markdown("### Highest Risk Entities")
    entity_risk = scored.groupby("entity")["composite_score"].max().sort_values(ascending=False).head(10)
    short_names = [e.split("(")[0].strip()[:35] for e in entity_risk.index]
    fig4 = go.Figure(go.Bar(
        x=entity_risk.values[::-1],
        y=short_names[::-1],
        orientation="h",
        marker_color=["#c0392b" if s >= 50 else "#f5a623" for s in entity_risk.values[::-1]],
        text=[f"{s:.1f}" for s in entity_risk.values[::-1]],
        textposition="outside",
        textfont=dict(color="#f0ece2")
    ))
    fig4.add_vline(x=50, line_dash="dash", line_color="#c0392b")
    fig4.update_layout(
        title="Top 10 Entities by Maximum Risk Score",
        title_font_color="#f5a623",
        paper_bgcolor="#242424",
        plot_bgcolor="#242424",
        font_color="#f0ece2",
        xaxis=dict(gridcolor="#333333", title="Max Composite Risk Score"),
        yaxis=dict(gridcolor="#333333"),
        height=420
    )
    st.plotly_chart(fig4, use_container_width=True)

    st.markdown("""
    <div class="footer">
        Built on public Ghanaian data · Open source · Grounded in Public Procurement Act 663<br/>
        Built for Ghana. Accountable to every Ghanaian. · Ghana AI Summit 2026
    </div>
    """, unsafe_allow_html=True)

# ══════════════════════════════════════════
# PAGE 2 — FLAGGED CONTRACTS
# ══════════════════════════════════════════
elif page == "🔴 Flagged Contracts":

    st.markdown("# 🔴 Flagged Contracts")
    st.markdown("Every contract the system has escalated or flagged for review — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False).reset_index(drop=True)

    col1, col2, col3 = st.columns(3)
    col1.metric("Contracts shown", len(filtered))
    col2.metric("Confirmed fraud in view", int(filtered["fraud_label"].sum()))
    col3.metric("Avg risk score", f"{filtered['composite_score'].mean():.1f}")

    st.markdown("<br/>", unsafe_allow_html=True)

    # Styled table
    def style_tier(val):
        if "ESCALATE" in str(val):
            return "background-color:#2a1a1a; color:#c0392b; font-weight:700;"
        elif "REVIEW" in str(val):
            return "background-color:#2a2410; color:#f5a623; font-weight:700;"
        return "color:#2d6a4f;"

    def style_fraud(val):
        if val == 1:
            return "color:#c0392b; font-weight:700;"
        return "color:#2d6a4f;"

    display_df = filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].copy()
    display_df.columns = ["Entity", "Supplier", "Risk Score", "Tier", "Fraud Confirmed"]
    display_df["Fraud Confirmed"] = display_df["Fraud Confirmed"].map({1: "✅ YES", 0: "⬜ No"})

    st.dataframe(
        display_df,
        use_container_width=True,
        height=500
    )

    st.markdown("""
    <div class="footer">
        Built on public Ghanaian data · Open source · Grounded in Public Procurement Act 663
    </div>
    """, unsafe_allow_html=True)

# ══════════════════════════════════════════
# PAGE 3 — ENTITY SCORECARDS
# ══════════════════════════════════════════
elif page == "🏛️ Entity Scorecards":

    st.markdown("# 🏛️ Entity Integrity Scorecards")
    st.markdown("Select any public institution to see their full procurement integrity profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Search institution:", entity_list)

    ec = scored[scored["entity"] == selected_entity]
    max_score = ec["composite_score"].max()
    fraud_count = int(ec["fraud_label"].sum())
    total = len(ec)
    escalate_count = len(ec[ec["tier"] == "🔴 ESCALATE"])
    review_count = len(ec[ec["tier"] == "🟡 REVIEW"])
    grade, grade_class = get_grade(max_score)

    st.markdown("<br/>", unsafe_allow_html=True)

    col1, col2 = st.columns([3, 1])
    with col1:
        col_a, col_b, col_c, col_d, col_e = st.columns(5)
        col_a.metric("Total Contracts", total)
        col_b.metric("Max Risk Score", f"{max_score:.1f}")
        col_c.metric("Confirmed Fraud", fraud_count)
        col_d.metric("🔴 Escalated", escalate_count)
        col_e.metric("🟡 Review", review_count)
    with col2:
        st.markdown(f"""
        <div style="text-align:center; padding: 8px;">
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Integrity Grade</div>
            <div style="margin-top:8px;"><span class="{grade_class}">{grade}</span></div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br/>", unsafe_allow_html=True)

    # Risk trend by contract
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(range(len(ec))),
        y=ec["composite_score"].values,
        mode="lines+markers",
        line=dict(color="#f5a623", width=2),
        marker=dict(
            color=["#c0392b" if t == "🔴 ESCALATE" else "#f5a623" if t == "🟡 REVIEW" else "#2d6a4f" for t in ec["tier"]],
            size=10
        ),
        name="Risk Score"
    ))
    fig.add_hline(y=50, line_dash="dash", line_color="#c0392b", annotation_text="ESCALATE")
    fig.add_hline(y=35, line_dash="dash", line_color="#f5a623", annotation_text="REVIEW")
    fig.update_layout(
        title=f"Risk Score Profile — {selected_entity.split('(')[0].strip()}",
        title_font_color="#f5a623",
        paper_bgcolor="#242424",
        plot_bgcolor="#242424",
        font_color="#f0ece2",
        yaxis=dict(gridcolor="#333333", range=[0, 100], title="Risk Score"),
        xaxis=dict(gridcolor="#333333", title="Contract")
    )
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("### All contracts for this entity:")
    display_ec = ec[["supplier", "composite_score", "tier", "fraud_label"]].copy()
    display_ec.columns = ["Supplier", "Risk Score", "Tier", "Fraud Confirmed"]
    display_ec["Fraud Confirmed"] = display_ec["Fraud Confirmed"].map({1: "✅ YES", 0: "⬜ No"})
    st.dataframe(
        display_ec.sort_values("Risk Score", ascending=False).reset_index(drop=True),
        use_container_width=True
    )

    st.markdown("""
    <div class="footer">
        Built on public Ghanaian data · Open source · Grounded in Public Procurement Act 663
    </div>
    """, unsafe_allow_html=True)

# ══════════════════════════════════════════
# PAGE 4 — CONTRACT DEEP DIVE
# ══════════════════════════════════════════
elif page == "🔍 Contract Deep Dive":

    st.markdown("# 🔍 Contract Deep Dive")
    st.markdown("Select any contract for a complete risk breakdown — flags, legal citations, and AI explanation.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:45]} | Score: {scored.loc[i, 'composite_score']} | {scored.loc[i, 'tier']}"
    )

    row = scored.loc[contract_idx]
    score_val = float(row["composite_score"])
    grade, grade_class = get_grade(score_val)

    col1, col2 = st.columns([1.2, 1])
    with col1:
        tier_color = "#c0392b" if "ESCALATE" in row["tier"] else "#f5a623" if "REVIEW" in row["tier"] else "#2d6a4f"
        st.markdown(f"""
        <div style="background:#242424; border:1px solid #333; border-radius:8px; padding:24px;">
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Entity</div>
            <div style="color:#f0ece2; font-size:1rem; font-weight:700; margin-bottom:12px;">{row['entity']}</div>
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Supplier</div>
            <div style="color:#f0ece2; font-size:1rem; margin-bottom:12px;">{row['supplier']}</div>
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Risk Tier</div>
            <div style="color:{tier_color}; font-size:1.1rem; font-weight:900; margin-bottom:12px;">{row['tier']}</div>
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Fraud Confirmed</div>
            <div style="font-size:1rem; font-weight:700;">{"✅ YES — Confirmed by Auditor-General" if row['fraud_label'] == 1 else "⬜ Not confirmed in available records"}</div>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=score_val,
            number={"font": {"color": "#f0ece2", "size": 48}},
            title={"text": "Composite Risk Score", "font": {"color": "#f5a623", "size": 14}},
            gauge={
                "axis": {"range": [0, 100], "tickcolor": "#f0ece2"},
                "bar": {"color": "#c0392b" if score_val >= 50 else "#f5a623" if score_val >= 35 else "#2d6a4f"},
                "bgcolor": "#242424",
                "bordercolor": "#333333",
                "steps": [
                    {"range": [0, 35], "color": "#1a2a1a"},
                    {"range": [35, 50], "color": "#2a2410"},
                    {"range": [50, 100], "color": "#2a1a1a"}
                ],
                "threshold": {
                    "line": {"color": "#f0ece2", "width": 3},
                    "thickness": 0.8,
                    "value": score_val
                }
            }
        ))
        fig.update_layout(
            paper_bgcolor="#242424",
            font_color="#f0ece2",
            height=280,
            margin=dict(t=40, b=0)
        )
        st.plotly_chart(fig, use_container_width=True)
        st.markdown(f"""
        <div style="text-align:center; margin-top:-16px;">
            <span class="{grade_class}">Grade {grade}</span>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br/>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("### ⚠️ Flags Raised")
        flags = parse_list(row["flags"])
        if flags:
            for f in flags:
                st.markdown(f'<div class="flag-card">⚠️ {f}</div>', unsafe_allow_html=True)
        else:
            st.markdown("No flags raised.")

    with col2:
        st.markdown("### ⚖️ Legal Citations")
        citations = parse_list(row["legal_citations"])
        if citations:
            for c in citations:
                st.markdown(f'<div class="citation-card">⚖️ {c}</div>', unsafe_allow_html=True)
        else:
            st.markdown("No citations.")

    with col3:
        st.markdown("### 🤖 AI Explanation")
        shap_items = parse_list(row["shap_top3"])
        if shap_items:
            for s in shap_items:
                st.markdown(f'<div class="shap-card">🔍 {s}</div>', unsafe_allow_html=True)
        else:
            st.markdown("No SHAP data.")

    st.markdown("""
    <div class="footer">
        Built on public Ghanaian data · Open source · Grounded in Public Procurement Act 663<br/>
        Built for Ghana. Accountable to every Ghanaian. · Ghana AI Summit 2026
    </div>
    """, unsafe_allow_html=True)
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard rewritten — Ghana identity design")
print("⏳ Rerun Cell 5 to relaunch.")

# Git: Cell 13 — full dashboard redesign Ghana identity

✅ Dashboard rewritten — Ghana identity design
⏳ Rerun Cell 5 to relaunch.


In [17]:
# Cell 14 — Definitive dashboard launcher with full URL extraction

import subprocess
import threading
import time

# Step 1 — Reinstall streamlit
subprocess.run(["pip", "install", "streamlit", "-q"])

# Step 2 — Install cloudflared
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
])
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
print("✅ Dependencies ready")

# Step 3 — Launch Streamlit in background
def run_streamlit():
    subprocess.run([
        "/usr/local/bin/streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()
time.sleep(8)
print("✅ Streamlit running")

# Step 4 — Start cloudflared
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Step 5 — Read ALL output lines until URL appears (from cell 5b fix)
print("⏳ Waiting for URL...")
for _ in range(60):
    line = tunnel.stderr.readline().decode("utf-8").strip()
    if line:
        print(line)
    if "trycloudflare.com" in line and "http" in line:
        print("\n" + "=" * 55)
        print("👁️  EYE OF RA IS LIVE — copy the URL above")
        print("=" * 55)
        break
    time.sleep(1)

# Git: Cell 14 — definitive launcher with full URL extraction

✅ Dependencies ready
✅ Streamlit running
⏳ Getting URL...
👁️  EYE OF RA IS LIVE

🌐 trycloudflare.com...

No password needed.


In [16]:
# Cell 15 — Fix syntax error in app.py (escaped newlines in chart labels)

content = open("/content/app.py").read()
content = content.replace(
    'x=["🔴 ESCALATE\n(caught)", "🟡 REVIEW\n(caught)", "🟢 MONITOR\n(missed)"]',
    'x=["🔴 ESCALATE (caught)", "🟡 REVIEW (caught)", "🟢 MONITOR (missed)"]'
)
with open("/content/app.py", "w") as f:
    f.write(content)

print("✅ Syntax error fixed")
print("⏳ Rerun Cell 14 to launch.")

# Git: Cell 15 — fix syntax error in app.py

✅ Syntax error fixed
⏳ Rerun Cell 14 to launch.


In [19]:
# Cell 16 — Remove integrity grades from app.py

content = open("/content/app.py").read()

# Remove get_grade function
content = content.replace("""
def get_grade(score):
    if score >= 60: return "F", "grade-f"
    elif score >= 45: return "D", "grade-d"
    elif score >= 30: return "C", "grade-c"
    elif score >= 15: return "B", "grade-b"
    else: return "A", "grade-a"
""", "")

# Remove grade CSS classes
content = content.replace("""    .grade-f { background:#c0392b; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-d { background:#e67e22; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-c { background:#f5a623; color:#1a1a1a; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-b { background:#2d6a4f; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
    .grade-a { background:#1a7a4a; color:white; padding:4px 12px; border-radius:4px; font-weight:900; font-size:1.4rem; }
""", "")

# Remove grade from entity scorecard
content = content.replace("""    with col2:
        st.markdown(f\"\"\"
        <div style="text-align:center; padding: 8px;">
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Integrity Grade</div>
            <div style="margin-top:8px;"><span class="{grade_class}">{grade}</span></div>
        </div>
        \"\"\", unsafe_allow_html=True)""",
"""    with col2:
        st.markdown(f\"\"\"
        <div style="text-align:center; padding: 8px;">
            <div style="color:#aaaaaa; font-size:0.7rem; text-transform:uppercase; letter-spacing:1px;">Max Risk Score</div>
            <div style="color:#f5a623; font-size:2rem; font-weight:900; margin-top:8px;">{max_score:.1f}</div>
            <div style="color:#aaaaaa; font-size:0.7rem;">out of 100</div>
        </div>
        \"\"\", unsafe_allow_html=True)""")

# Remove grade from deep dive
content = content.replace("""        st.markdown(f\"\"\"
        <div style="text-align:center; margin-top:-16px;">
            <span class="{grade_class}">Grade {grade}</span>
        </div>
        \"\"\", unsafe_allow_html=True)""", "")

# Remove grade variable assignments
content = content.replace("    grade, grade_class = get_grade(score_val)\n", "")
content = content.replace("    grade, grade_class = get_grade(max_score)\n", "")

with open("/content/app.py", "w") as f:
    f.write(content)

print("✅ Integrity grades removed")
print("⏳ Rerun Cell 14 to relaunch.")

# Git: Cell 16 — remove deterministic integrity grades

✅ Integrity grades removed
⏳ Rerun Cell 14 to relaunch.
